In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS Employee_data

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS employee_data.datset_loading

In [0]:
display(
    dbutils.fs.ls("/Volumes/workspace/employee_project/employee_files/")
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/employee_project/employee_files/employees.csv,employees.csv,2114,1786961268000


In [0]:
file_path='/Volumes/workspace/employee_project/employee_files/employees.csv'
employee_df=spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

In [0]:
display(employee_df)

employee_id,first_name,last_name,department,designation,manager_id,salary,joining_date,city,employee_status
EMP100001,Devansh,Thakkar,Sales,Manager,null,578082,2025-11-26,Chennai,Active
EMP100002,Yashoda,Batra,Sales,Engineer,EMP100001,372132,2024-01-29,Chennai,Inactive
EMP100003,Meera,Banerjee,Sales,Engineer,EMP100001,598830,2025-08-25,Bengaluru,Active
EMP100004,Agastya,Chana,Sales,Engineer,EMP100001,1174244,2021-12-16,Bengaluru,Active
EMP100005,Nachiket,Bath,HR,Manager,EMP100001,-55000,2025-02-14,Hyderabad,Inactive
EMP100006,Avni,Prakash,IT,Manager,EMP100001,1156795,2023-02-22,Hyderabad,Active
EMP100007,Harinakshi,Kota,HR,Manager,EMP100001,408618,2024-02-21,Pune,Inactive
EMP100008,Neha,Dhillon,Sales,Engineer,EMP100001,421035,2022-09-01,Pune,Active
EMP100009,Balvan,Palla,IT,Manager,EMP100001,880097,2022-06-11,Mumbai,Active
EMP100010,Wriddhish,Dugar,Sales,Engineer,EMP100001,595442,2026-04-23,Mumbai,Inactive


In [0]:
employee_df.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- designation: string (nullable = true)
 |-- manager_id: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- joining_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- employee_status: string (nullable = true)



In [0]:
print(f"Total Number of employees present in employees.csv file is:",employee_df.count())

Total Number of employees present in employees.csv file is: 25


### Answering Bussiness question
> - How many employees are currently in each department? 
> - What is the average salary by department? 
> - Which cities have the highest number of employees? 
> - How many employees are active or inactive? 
> - Who reports to which manager? 
> - Are there duplicate employee records? 
> - Are employee salaries valid? 
> - Are required fields missing? 
> - Are Employee Salaries Valid?


In [0]:
department_df = employee_df.groupBy('department').count().orderBy("count",ascending=False).withColumnRenamed("count","Employee_count")
display(department_df)

department,Employee_count
Sales,12
IT,6
HR,5
Finance,2


In [0]:
# Now we will perform using agg.This can perform in the same way like previous one
from pyspark.sql.functions import avg
average_salary_department=employee_df.groupBy('department').agg(avg("salary").alias("Average_Salary")).orderBy("Average_Salary",ascending=False)
display(average_salary_department)

department,Average_Salary
IT,619138.8333333334
Sales,536391.0833333334
HR,488405.0
Finance,328183.5


In [0]:
employee_count=employee_df.groupBy("city").count().orderBy("count",ascending=False)
employee_count.show(n=5)

+----------+-----+
|      city|count|
+----------+-----+
|   Chennai|    4|
|Coimbatore|    3|
| Hyderabad|    3|
|     Kochi|    3|
|     Delhi|    3|
+----------+-----+
only showing top 5 rows


In [0]:
status_count=employee_df.groupBy("employee_status").count()
display(status_count)

employee_status,count
Inactive,11
Active,14


In [0]:
# Let as first inspect the relationship
employee_df.select(
    "employee_id",
    "first_name",
    "last_name",
    "manager_id",
    "department"
).display()

employee_id,first_name,last_name,manager_id,department
EMP100001,Devansh,Thakkar,null,Sales
EMP100002,Yashoda,Batra,EMP100001,Sales
EMP100003,Meera,Banerjee,EMP100001,Sales
EMP100004,Agastya,Chana,EMP100001,Sales
EMP100005,Nachiket,Bath,EMP100001,HR
EMP100006,Avni,Prakash,EMP100001,IT
EMP100007,Harinakshi,Kota,EMP100001,HR
EMP100008,Neha,Dhillon,EMP100001,Sales
EMP100009,Balvan,Palla,EMP100001,IT
EMP100010,Wriddhish,Dugar,EMP100001,Sales


In [0]:
# Who reports to which manager?
# For this question we need to perform the join to manager name based on employee_id and manager id
# same table splits into two different table name
from pyspark.sql.functions import col
employees = employee_df.alias("employee")
managers = employee_df.alias("manager")

employee_manager_df=employees.join(
    managers,
    col("employee.manager_id")== col("manager.employee_id"),"left"
).select(
    col("employee.employee_id").alias("employee_id"),
    col("employee.first_name").alias("employee_first_name"),
    col("employee.last_name").alias("employee_last_name"),
    col("employee.department").alias("department"),
    col("employee.manager_id").alias("manager_id"),
    col("manager.first_name").alias("manager_first_name"),
    col("manager.last_name").alias("manager_last_name")
)
employee_manager_df.show(5)

+-----------+-------------------+------------------+----------+----------+------------------+-----------------+
|employee_id|employee_first_name|employee_last_name|department|manager_id|manager_first_name|manager_last_name|
+-----------+-------------------+------------------+----------+----------+------------------+-----------------+
|  EMP100001|            Devansh|           Thakkar|     Sales|      NULL|              NULL|             NULL|
|  EMP100002|            Yashoda|             Batra|     Sales| EMP100001|           Devansh|          Thakkar|
|  EMP100003|              Meera|          Banerjee|     Sales| EMP100001|           Devansh|          Thakkar|
|  EMP100004|            Agastya|             Chana|     Sales| EMP100001|           Devansh|          Thakkar|
|  EMP100005|           Nachiket|              Bath|        HR| EMP100001|           Devansh|          Thakkar|
+-----------+-------------------+------------------+----------+----------+------------------+-----------

In [0]:
employee_id_count=employee_df.groupBy("employee_id").count()
duplicate_employee=employee_id_count.filter(col("count")>1)
duplicate_employee.show()

+-----------+-----+
|employee_id|count|
+-----------+-----+
+-----------+-----+



In [0]:
# check missing employee IDs

employee_df.filter(
    col("employee_id").isNull()
).count()

# Missing department

employee_df.filter(
    col("department").isNull()
).count()

# Missing salary

employee_df.filter(
    col("salary").isNull()
).count()

0

In [0]:
invalid_salary=employee_df.filter(col("salary")<0)
invalid_salary.show()

+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+
|employee_id|first_name|last_name|department|designation|manager_id|salary|joining_date|     city|employee_status|
+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+
|  EMP100005|  Nachiket|     Bath|        HR|    Manager| EMP100001|-55000|  2025-02-14|Hyderabad|       Inactive|
|  EMP100012|      Owen|   Sankar|     Sales|    Manager| EMP100001|-72000|  2022-11-12|    Delhi|       Inactive|
|  EMP100020|       Avi|  Chander|        IT|    Analyst| EMP100001|-48000|  2024-05-24|     Pune|         Active|
|  EMP100025|     Sneha|   Munshi|        IT|   Engineer| EMP100001|-65000|  2026-05-08|  Chennai|       Inactive|
+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+



In [0]:
employee_df.createOrReplaceTempView("employee_data")

### Python --> SQL

In [0]:
%sql
select * from employee_data;

employee_id,first_name,last_name,department,designation,manager_id,salary,joining_date,city,employee_status
EMP100001,Devansh,Thakkar,Sales,Manager,null,578082,2025-11-26,Chennai,Active
EMP100002,Yashoda,Batra,Sales,Engineer,EMP100001,372132,2024-01-29,Chennai,Inactive
EMP100003,Meera,Banerjee,Sales,Engineer,EMP100001,598830,2025-08-25,Bengaluru,Active
EMP100004,Agastya,Chana,Sales,Engineer,EMP100001,1174244,2021-12-16,Bengaluru,Active
EMP100005,Nachiket,Bath,HR,Manager,EMP100001,-55000,2025-02-14,Hyderabad,Inactive
EMP100006,Avni,Prakash,IT,Manager,EMP100001,1156795,2023-02-22,Hyderabad,Active
EMP100007,Harinakshi,Kota,HR,Manager,EMP100001,408618,2024-02-21,Pune,Inactive
EMP100008,Neha,Dhillon,Sales,Engineer,EMP100001,421035,2022-09-01,Pune,Active
EMP100009,Balvan,Palla,IT,Manager,EMP100001,880097,2022-06-11,Mumbai,Active
EMP100010,Wriddhish,Dugar,Sales,Engineer,EMP100001,595442,2026-04-23,Mumbai,Inactive


In [0]:
%sql
select count(*) as total_employees from employee_data;

total_employees
25


In [0]:
%sql
select distinct(department) from employee_data

department
Sales
IT
HR
Finance


In [0]:
%sql
select department,count(*) as employee_count from employee_data
group by department
order by employee_count desc;

department,employee_count
Sales,12
IT,6
HR,5
Finance,2


In [0]:
%sql
select department,AVG(salary) as average_salary
from employee_data
group by department
order by average_salary desc;

department,average_salary
IT,619138.8333333334
Sales,536391.0833333334
HR,488405.0
Finance,328183.5


In [0]:
%sql
select city,count(*) as employee_count from employee_data
group by city
order by employee_count desc;

city,employee_count
Chennai,4
Delhi,3
Kochi,3
Hyderabad,3
Coimbatore,3
Pune,3
Mumbai,3
Bengaluru,3


In [0]:
%sql
select employee_status,count(*)
from employee_data
group by employee_status;

employee_status,count(*)
Inactive,11
Active,14


In [0]:
%sql
SELECT *
FROM employee_data
WHERE salary <= 0;

employee_id,first_name,last_name,department,designation,manager_id,salary,joining_date,city,employee_status
EMP100005,Nachiket,Bath,HR,Manager,EMP100001,-55000,2025-02-14,Hyderabad,Inactive
EMP100012,Owen,Sankar,Sales,Manager,EMP100001,-72000,2022-11-12,Delhi,Inactive
EMP100020,Avi,Chander,IT,Analyst,EMP100001,-48000,2024-05-24,Pune,Active
EMP100025,Sneha,Munshi,IT,Engineer,EMP100001,-65000,2026-05-08,Chennai,Inactive


In [0]:
%sql
select * from employee_data
where employee_id is null or
department is null;

employee_id,first_name,last_name,department,designation,manager_id,salary,joining_date,city,employee_status


In [0]:
%sql
select employee_id,count(*) 
from employee_data
group by employee_id
having count(*) > 1;

employee_id,count(*)


## Employee Distribution by Department

The HR team wants to understand how the workforce
is distributed across different departments.

The following query counts the number of employees
in each department.